# Day 2: Identifying "Agentic AI" Potential Workflows

## Goal of the Day
By the end of this session, you will:
- Recap and solidfy the core concept of this Weekend: **LLM + Tool = Agentic AI**
- Identify potential agentic workflows in company settings
- Learn how to build and integrate tools with LLMs
- Understand orchestration patterns for complex tool interactions

## Notebook Structure
1. **Part 1 - Tools**: Identify and Build Wrappers for LLMs
2. **Part 2 - Orchestration**: Complex Tool Use Patterns
3. **Part 3 - Integration**: Automatic Dashboard from Database

---
## Setup

First, let's clone the course repository from GitHub and install the required dependencies.

In [ ]:
# Clone the course repo from GitHub (public repo)
import os, sys
from google.colab import userdata

REPO_URL = "https://github.com/MAS-AID-AI-Project/weekend4_agentic_public.git"

# Clone only if not already cloned
if not os.path.exists('/content/weekend4_agentic_public'):
    !git clone {REPO_URL} /content/weekend4_agentic_public
else:
    # Pull latest changes
    !cd /content/weekend4_agentic_public && git pull

# Navigate to the project directory and add to Python path for imports
PROJECT_DIR = '/content/weekend4_agentic_public/project_saturday'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Repository cloned and navigated to project directory!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and navigated to project directory!


In [ ]:
# Install requirements
!pip install -q openai ipywidgets

print("Dependencies installed!")

In [ ]:
# Import libraries
from openai import OpenAI
import json
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

# Import our tool implementations
import tools
from resources import flow_exercise_helper

import importlib
importlib.reload(tools)
importlib.reload(flow_exercise_helper)


In [ ]:
# Setup OpenRouter API client
# IMPORTANT: Add your OpenRouter API key as a Colab secret named 'OPENROUTER_API_KEY'
# Go to the 🔑 icon in the left sidebar → Add a secret named OPENROUTER_API_KEY

from google.colab import userdata
API_KEY = userdata.get('OPENROUTER_API_KEY')

if not API_KEY:
    print("WARNING --- Please add your OpenRouter API key as a Colab secret named OPENROUTER_API_KEY!")
else:
    print("API key loaded successfully!")

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
)

In [ ]:
# Setup demo database
db_conn = tools.setup_demo_database()
print("Demo database created with sample customers, products, and sales!")

Demo database created with sample customers, products, and sales!


---
# Part 1: Tool Identification

## The Core Concept: LLM + Tool = Agentic AI

Agentic AI is built on a simple but powerful principle:

```
LLM (Language Model) + Tool (Function/API) = Agentic AI
```

- **LLM**: Understands natural language, makes decisions, plans actions
- **Tool**: Performs specific actions (query database, send email, call API)
- **Agentic AI**: Combines both to autonomously complete complex tasks

Let's start by identifying agentic workflows in real business scenarios.

## 1.1: Identifying Agentic Flows - Interactive Exercises

Before building agentic workflows, let's understand the common building blocks that make them work.

### The Agentic Toolbox: What's Under the Hood?

Every agentic AI system is built from a combination of these components:

#### Entry & Exit Points (Triggers and Delivery)
- **Incoming Mailbox / User Prompt**: Where requests enter the system
- **Outbox**: Where responses are delivered back to users

#### The Reasoning Core
- **LLM Agent**: The "brain" that interprets requests, makes decisions, and orchestrates tool usage

#### Knowledge & Data
- **Technical Manuals (RAG)**: Retrieval-Augmented Generation - searches through documents and knowledge bases
- **Database (SQL) - Specialized Databases ends with 'DB'**: Structured data storage for products, customers, sales, etc.
- **Web Search**: External search engines for public information
- **Public Financial News**: Special search for external news feeds and financial information

#### Specialized Capabilities
- **Authentication Layer**: Security checkpoint ensuring only authorized users can access the system
- **Plotting Tool**: Creates visualizations like charts and graphs
- **Creative Image Gen (DALL-E)**: AI-powered image generation for creative content

---

Now let's practice identifying the correct flow of components in two real business scenarios. Click the boxes in the order you think they should be used! The first one is already given.

**Note**: You can click a selected box again to unselect it. Not all boxes belong in the workflow! Some might be incorrect choices.

### Exercise 1: The 'Auto-Pilot' Customer Support Agent

**Business Context:** Your company, _TechNova_, sells complex industrial sensors. Traditionally, a human assigned to client support spends 20 minutes looking up help manuals and personal warranty for many client emails. You are designing an **Agentic Support Flow** to automate this.

**Business Objective:** Resolve a technical query in under 60 seconds without human intervention.

**Scenario:** A client emails: _"My PX-100 sensor is flashing red. Is this covered under my Gold Support plan?"_

**The Goal:** Select the boxes in the order that the Agentic Support Flow would take place.

**Remember**: The trigger (Incoming Mailbox) is already selected. Continue from there!

In [ ]:
# Run this cell to display the interactive exercise
flow_exercise_helper.display_exercise_1()

### Exercise 2: The 'Sales Intelligence' Dashboard Agent

**Business Context:** The VP of Sales is preparing for a quarterly review. Instead of asking a Data Analyst to spend two days building a report, they want to ask the system directly in plain English.

**Business Objective:** Transform a natural language question into a live, visual data insight.

**Scenario:** The Admin types: _"Show me a bar chart of 'PX-300' sales trends over the last 12 months."_

**The Goal:** Arrange the Agentic Flow that would allow the LLM to **safely** talk to your company's **private** database.

**Hint:** To access the Secure DB you need to first get an authentication flag for the user!

**Remember**: The trigger (User Prompt) is already selected. Continue from there!

In [ ]:
# Run this cell to display the second exercise
flow_exercise_helper.display_exercise_2()

---
## 1.2: Understanding Function Calling in LLMs

### What is Function Calling?

**Function calling** (also known as tool use) is a standardized way to describe tools and their capabilities to LLMs. Think of it as a "contract" that tells the LLM:

1. **What the tool does** (description)
2. **What inputs it needs** (parameters)
3. **What types those inputs should be** (schema)

**Recall the Langchain notebook** from yesterday? This is what was generated automatically for us by the `@tool` decorator. However recall again that this decorator is not all powerful, and doesn't directly give you constraints!



### Why is Function Calling Important? - Recap from Yesterday

- **Discoverability**: LLMs can understand what tools are available
- **Type Safety**: Ensures correct data types are used
- **Automation**: Enables LLMs to use tools without hardcoded logic
- **Standardization**: Common format across different AI systems

### Function Declaration Format

Tools are described using function declarations that define:
- `name`: The function name
- `description`: What the function does
- `parameters`: Object containing:
  - `type`: The data type (string, integer, object, array, number)
  - `properties`: Each parameter's specification
  - `description`: What each parameter is for

### Exercise 1: The "Sales Intelligence" Schema

**Context:** Our Agent is now connected to the live Sales Database. However, databases are expensive to query! If the Agent asks for 50 years of data at once, it could crash the system or run up a massive cloud bill. We need to set "guardrails" on the tool.

**Business Objective:** Protect system resources by limiting how much data the AI can request and ensuring it always specifies a product category.

**Your Task:** Complete the schema below to enforce these three business rules:

1. **Safety**: The `limit_rows` must be a whole number (integer) to avoid database errors.
2. **Cost Control**: The `time_period_months` cannot exceed 12 to prevent massive, slow queries.
3. **Strictness**: Every request must include the `product_category` so the report isn't too broad.

In [ ]:
# Exercise: Complete the Tool Schema for the Sales Dashboard
# Available types/constraints: "string", "integer", "number", "required", "maximum"

sales_chart_schema = {
    "name": "generate_sales_chart",
    "description": "Generates a bar chart based on product sales history.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_category": {
                "type": "???",  # TODO: Which type represents a word like 'Red Paper'?
                "description": "The specific category of product to analyze."
            },
            "time_period_months": {
                "type": "integer",
                "???": 12,      # TODO: Use a constraint to ensure the AI doesn't ask for more than 12 months.
                "description": "Duration of the trend analysis in months."
            },
            "limit_rows": {
                "type": "???",  # TODO: What type should a row count be?
                "description": "The maximum number of data rows to process."
            }
        },
        "???": ["product_category"] # TODO: Ensure the Agent ALWAYS provides a product category.
    }
}

# Print the schema to see your work
print("Your Sales Chart Schema:")
print(json.dumps(sales_chart_schema, indent=2))

Your Sales Chart Schema:
{
  "name": "generate_sales_chart",
  "description": "Generates a bar chart based on product sales history.",
  "parameters": {
    "type": "object",
    "properties": {
      "product_category": {
        "type": "???",
        "description": "The specific category of product to analyze."
      },
      "time_period_months": {
        "type": "integer",
        "???": 12,
        "description": "Duration of the trend analysis in months."
      },
      "limit_rows": {
        "type": "???",
        "description": "The maximum number of data rows to process."
      }
    },
    "???": [
      "product_category"
    ]
  }
}


In [ ]:
# Check your answer
tools.verify_sales_chart_schema(sales_chart_schema)

### Exercise 2: The "Support Ticket" Schema

**Context:** When our Agent identifies that a problem is too complex, it needs to hand the case over to a human by creating a "Support Ticket." We need to make sure the Agent categorizes the ticket correctly so it lands in the right department's inbox.

**Business Objective:** Prevent the AI from making up its own priority levels (e.g., "Mega-Urgent") and ensure it always provides a clear summary for the human staff.

**Your Task:** Complete the schema to enforce these rules:

1. **Fixed Options**: The `priority` must be limited to a specific list (Enum).
2. **Clarity**: Provide a description that tells the LLM exactly what a "Summary" should look like.
3. **Completeness**: Make both fields "Required" so the human agent isn't left guessing.

In [ ]:
# Exercise: Complete the Support Ticket Schema
# Available types/constraints: "string", "enum", "required", "description"

support_ticket_schema = {
    "name": "create_human_support_ticket",
    "description": "Escalates a complex issue to a human technical representative.",
    "parameters": {
        "type": "object",
        "properties": {
            "priority": {
                "type": "string",
                "???": ["low", "medium", "high"], # TODO: What attribute restricts a string to these 3 choices?
                "description": "The urgency level of the ticket."
            },
            "issue_summary": {
                "type": "string",
                "???": "A concise, 1-sentence summary of the technical failure." # TODO: Add the attribute that explains the 'Role' of this field to the AI.
            }
        },
        "???": ["priority", "issue_summary"] # TODO: Ensure the Agent provides BOTH pieces of info.
    }
}

# Print the schema to see your work
print("Your Support Ticket Schema:")
print(json.dumps(support_ticket_schema, indent=2))

In [ ]:
# Check your answer
tools.verify_support_ticket_schema(support_ticket_schema)

---
## 1.3: Testing Tools in Action - The Robustness Challenge

Now that we understand how to define tools with function schemas, let's see them in action! We'll test our tools with real LLM calls and discover an important limitation.

### Our Tools Are Ready

Behind the scenes, we have two tools connected to our demo database:
- **`customer_lookup`**: Queries the customers table by ID or email
- **`product_inventory`**: Queries the products table by ID, name, or category

Let's test them with natural language queries!

In [ ]:
#@title Agentic Chat Wrapper { display-mode: "form" }

# Helper function to test our agentic workflow
def test_agentic_workflow(user_query):
    """
    Test the complete agentic workflow: User -> LLM -> Tools -> Response
    """
    print(f"User Query: {user_query}")
    print("=" * 60)

    # Step 1: Send query to LLM with available tools
    response = client.chat.completions.create(
        model="google/gemini-2.0-flash-001",
        messages=[{"role": "user", "content": user_query}],
        tools=tools.get_openai_function_declarations(),
        tool_choice="auto"
    )

    message = response.choices[0].message

    # Step 2: Check if the LLM wants to use a function
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        print(f"\nLLM wants to use function: {function_name}")
        print(f"Parameters: {function_args}")

        # Step 3: Execute the function
        function_result = tools.execute_tool(db_conn, function_name, function_args)

        print(f"\nFunction Result:")
        print(f"  {json.dumps(function_result, indent=2)}")

        # Step 4: Send function result back to LLM for final response
        follow_up_response = client.chat.completions.create(
            model="google/gemini-2.0-flash-001",
            messages=[
                {"role": "user", "content": user_query},
                message,
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(function_result)
                }
            ]
        )

        print(f"\nLLM's Final Response:")
        print(f"  {follow_up_response.choices[0].message.content}")

    else:
        print(f"\nLLM Response (no tool used):")
        print(f"  {message.content}")

    print("=" * 60)

### Test 1: A Simple Query That Works

Let's start with a straightforward query to see the system working correctly.

In [ ]:
# This query works perfectly - exact match!
test_agentic_workflow("What is the price and stock quantity of 'Red Paper'?")

### Test 2: The Robustness Problem

Now let's try the **exact same question** but with slightly different capitalization...

In [ ]:
# Same question, different capitalization - what happens?
test_agentic_workflow("What is the price and stock quantity of 'red paper'?")

### What Just Happened?

The query returned **zero results** just because of capitalization! The database has "Red Paper" but we asked for "red paper".

### Exercise: Try to Break It More!

Experiment with different variations. Can you find other ways the system fails?

In [ ]:
# TODO: Try different variations - typos, synonyms, partial names
# Examples to try:
#   - "red papper" (typo)
#   - "the red colored paper" (verbose)
#   - "paper that is red" (different phrasing)

test_agentic_workflow("???")

### Reality Check: Why Simple Tools Often Fail

In our exercises, we saw that even a "perfect" LLM can struggle when it meets a rigid database. This gap between **natural language** and **structured data** is the biggest hurdle in Agentic AI.

#### 1. The "Exact Match" Trap
Most corporate databases are literal. If the user asks for **"red paper"** but the database stores it as **"Red Paper"**, the system returns zero results.
* **Issues:** Case sensitivity, typos ("papper"), and synonyms ("crimson") cause the flow to break.

#### 2. The "Hallucination" Risk
LLMs are helpful, but they might hallucinate. Without strict boundaries, they might:
* **Invent Data:** Create a `customer_id` that doesn't exist (true colunm name is just `id`).
* **Guess Parameters:** Use a category like "Office Supplies" when the DB uses "Supplies_Office".
* **Misinterpret Intent:** Happens on complex queries, see part 2.

---

### Bridging the Gap: What's Next?

To move from a "classroom demo" to a **Production-Ready System**, we need more than just a single tool call. We need **Orchestration**.

In **Part 2**, we will explore how to build "loops" and "verification steps" that allow the Agent to:
* **Self-Correct:** If a query fails, try a fuzzy search.
* **Validate:** Check the database schema *before* executing a command.
* **Recover:** If the LLM makes a mistake, catch it and ask for a refinement.

**Next Step:** Ready to see how we turn these "fragile" tools into a robust system? Let's move on to **Part 2: Orchestration.**

---
# Part 2: Complex Tool Orchestration - XiYanSQL

Time to open the second notebook!

---
# Part 3: Putting It All Together - Automatic Dashboard Builder

## From Natural Language to Visual Insights

In Part 1, we learned how to connect an LLM to simple tools. In Part 2, we saw how orchestration makes those tools more robust. Now, we'll combine everything into a **complete pipeline** that transforms a natural language question into a dashboard.

### The Architecture

![Architecture](resources/architecture_p3.jpg)

**What we'll go through:**
1. Understand our database schema
2. Use our NL-to-SQL tool (as a black box from Part 2)
3. Transform query results into chart-friendly formats
4. Generate HTML dashboards to visualize our data

---
## 3.1: Understanding the Database Schema

Before we can build dashboards, we need to understand what data we have. Our demo database has three tables that work together:

### Visual Schema (from dbdiagram.io)

Below is a visual representation of our database schema. You can generate diagrams like this at [dbdiagram.io](https://dbdiagram.io) - it's a great tool for visualizing and documenting database structures!

![DB](resources/simple_db.jpg)

In [ ]:
# Display the database schema that the LLM will use
print(tools.get_db_schema_description())

### Exercise: What Can We Ask?

Given this schema, think about what questions are **possible** vs **impossible** to answer.

**For each scenario below, mark whether it's POSSIBLE ✅ or IMPOSSIBLE ❌:**

In [ ]:
# Exercise: For each question, replace "???" with "POSSIBLE" or "IMPOSSIBLE"
# Think about what data exists in our schema!

scenarios = {
    "1. Show total sales revenue per product": "???",
    "2. Show which customer bought the most items": "???",
    "3. Show sales trends over time (monthly)": "???",
    "4. Show customer satisfaction ratings": "???",
    "5. Show inventory levels by category": "???",
    "6. Show profit margins per product": "???",
    "7. Show sales by customer registration month": "???",
}

# Print your answers
print("="*60)
print("Schema Feasibility Analysis")
print("="*60)
for question, answer in scenarios.items():
    print(f"\n{question}")
    print(f"   Your answer: {answer}")

In [ ]:
# Run this cell to check your answers
tools.verify_schema_feasibility(scenarios)

---
## 3.2: Meet the Dashboard Visualization Tool

Now let's see what our dashboard tool can produce! The `display_dashboard()` function takes transformed data and generates beautiful HTML visualizations.

### What the Visualization Tool Expects

Sometimes you'll encounter more complicated workflows, where LLM can chain the output of one call with another without getting it back as **Text**.

The dashboard visualization would be such a tool, it doesn't work with raw SQL results. It needs **transformed data** in a specific format:

```python
# For a BAR chart:
{
    "chart_type": "bar",
    "labels": ["Product A", "Product B", "Product C"],
    "values": [150, 230, 180],
    "label_column": "product_name",
    "value_column": "total_sales"
}

# For a LINE chart:
{
    "chart_type": "line",
    "x": ["Jan", "Feb", "Mar", "Apr"],
    "y": [100, 150, 130, 200],
    "x_column": "month",
    "y_column": "sales"
}

# For a PIE chart:
{
    "chart_type": "pie",
    "labels": ["Category A", "Category B"],
    "values": [60, 40],
    "percentages": [60.0, 40.0],
    "total": 100
}
```

Let's see some examples!

In [ ]:
# Example 1: Bar Chart - Products by Stock Quantity
bar_data = {
    "chart_type": "bar",
    "labels": ["Blue Pen", "Notebook", "Paper Clips", "Red Paper", "Stapler"],
    "values": [500, 300, 200, 150, 75],
    "label_column": "product_name",
    "value_column": "stock_quantity"
}

tools.display_dashboard(bar_data, "Product Inventory Levels")

In [ ]:
# Example 2: Pie Chart - Sales by Category
pie_data = {
    "chart_type": "pie",
    "labels": ["Office Supplies", "Writing Instruments", "Office Equipment"],
    "values": [73, 8, 1],
    "percentages": [89.0, 9.8, 1.2],
    "total": 82
}

tools.display_dashboard(pie_data, "Sales Distribution by Category")

In [ ]:
# Example 3: Line Chart - Monthly Sales Trend
line_data = {
    "chart_type": "line",
    "x": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "y": [5, 10, 8, 0, 15, 0, 12, 7, 0, 10, 6, 0],
    "x_column": "month",
    "y_column": "units_sold"
}

tools.display_dashboard(line_data, "Red Paper Monthly Sales (2025)")

---
## 3.3: The Missing Link - Data Transformation

There's a problem: SQL queries return **raw tabular data**, but our visualization tool needs **chart-specific formats**.

### The Gap

```python
# What SQL gives us:
{
    "columns": ["product_name", "total_quantity"],
    "rows": [
        ["Red Paper", 73],
        ["Blue Pen", 8],
        ["Stapler", 1]
    ]
}

# What the chart needs:
{
    "chart_type": "bar",
    "labels": ["Red Paper", "Blue Pen", "Stapler"],
    "values": [73, 8, 1],
    ...
}
```

### The Solution: Transformation Functions

We have three transformation functions that bridge this gap:
- `transform_for_bar_chart(query_result, label_column, value_column)`
- `transform_for_line_chart(query_result, x_column, y_column)`
- `transform_for_pie_chart(query_result, label_column, value_column)`

Let's see how they work!

In [ ]:
# Example: Running a real SQL query and transforming the result

# Step 1: Execute a SQL query - Brain Teaser: Can you guess what this query is trying to achieve?
query = """
SELECT p.name as product_name, SUM(s.quantity) as total_sold
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_id
ORDER BY total_sold DESC
"""

query_result = tools.execute_sql_query(db_conn, query)
print("Raw SQL Result:")
print(json.dumps(query_result, indent=2))

In [ ]:
# Step 2: Transform for bar chart
chart_data = tools.transform_for_bar_chart(query_result, "product_name", "total_sold")
print("Transformed for Bar Chart:")
print(json.dumps(chart_data, indent=2))

In [ ]:
# Step 3: Display the dashboard!
tools.display_dashboard(chart_data, "Total Units Sold by Product")

### Exercise: Complete the Transformation

Below are two completed transformation examples. Your task is to complete the **third transformation** for a pie chart showing sales distribution by product category.

In [ ]:
# EXAMPLE 1: Bar chart of customer purchase counts (COMPLETED)
customer_query = """
SELECT c.name as customer_name, SUM(s.quantity) as items_bought
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY items_bought DESC
"""
customer_result = tools.execute_sql_query(db_conn, customer_query)
customer_chart = tools.transform_for_bar_chart(customer_result, "customer_name", "items_bought")
tools.display_dashboard(customer_chart, "Items Purchased by Customer")

In [ ]:
# EXAMPLE 2: Line chart of Red Paper sales over time (COMPLETED)
time_query = """
SELECT strftime('%Y-%m', sale_date) as month, SUM(quantity) as units_sold
FROM sales
WHERE product_id = 1
GROUP BY strftime('%Y-%m', sale_date)
ORDER BY month
"""
time_result = tools.execute_sql_query(db_conn, time_query)
time_chart = tools.transform_for_line_chart(time_result, "month", "units_sold")
tools.display_dashboard(time_chart, "Red Paper Sales Over Time")

In [ ]:
# EXERCISE 3: Pie chart of sales by product category (TODO: Complete this!)
# The SQL query is provided - you need to:
# 1. Execute it using tools.execute_sql_query
# 2. Transform it for a pie chart using tools.transform_for_pie_chart
# 3. Display it using tools.display_dashboard

category_query = """
SELECT p.category, SUM(s.quantity) as total_sold
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sold DESC
"""

# TODO: Execute the query
category_result = ???  # Hint: tools.execute_sql_query(db_conn, category_query)

# TODO: Transform for pie chart
category_chart = ???  # Hint: tools.transform_for_pie_chart(category_result, "???", "???")

# TODO: Display the dashboard
???  # Hint: tools.display_dashboard(category_chart, "???")

---
## 3.4: The Full Pipeline - Natural Language to Dashboard

Now let's put everything together! We'll use the `nl_to_sql()` function (our "black box" from Part 2) to convert natural language queries into SQL, then transform and visualize the results.

### The Complete Pipeline

```python
def create_dashboard_from_nl(user_question, chart_type, label_col, value_col, title):
    # Step 1: Natural Language -> SQL (Black box from Part 2)
    sql_query = tools.nl_to_sql(user_question)
    
    # Step 2: Execute SQL -> Raw Data
    query_result = tools.execute_sql_query(db_conn, sql_query)
    
    # Step 3: Transform Data -> Chart Format
    if chart_type == "bar":
        chart_data = tools.transform_for_bar_chart(query_result, label_col, value_col)
    elif chart_type == "pie":
        chart_data = tools.transform_for_pie_chart(query_result, label_col, value_col)
    # ... etc
    
    # Step 4: Visualize
    tools.display_dashboard(chart_data, title)
```

In [ ]:
# Let's test the full pipeline!

def create_dashboard_from_nl(user_question, chart_type="bar", title="Dashboard"):
    """
    Complete pipeline: Natural Language -> SQL -> Data -> Chart -> Display
    """
    print(f"User Question: {user_question}")
    print("="*60)
    
    # Step 1: Convert natural language to SQL
    print("\nStep 1: Converting to SQL...")
    sql_query = tools.nl_to_sql(user_question)
    print(f"Generated SQL:\n{sql_query}")
    
    # Step 2: Execute the query
    print("\nStep 2: Executing SQL...")
    query_result = tools.execute_sql_query(db_conn, sql_query)
    
    if not query_result["success"]:
        print(f"❌ Error: {query_result['error']}")
        return
    
    print(f"✅ Retrieved {query_result['row_count']} rows")
    print(f"Columns: {query_result['columns']}")
    
    # Step 3: Determine columns and transform
    print("\nStep 3: Transforming data...")
    columns = query_result["columns"]
    
    if len(columns) < 2:
        print("❌ Need at least 2 columns for a chart")
        return
    
    label_col = columns[0]  # First column = labels
    value_col = columns[1]  # Second column = values
    
    if chart_type == "bar":
        chart_data = tools.transform_for_bar_chart(query_result, label_col, value_col)
    elif chart_type == "pie":
        chart_data = tools.transform_for_pie_chart(query_result, label_col, value_col)
    elif chart_type == "line":
        chart_data = tools.transform_for_line_chart(query_result, label_col, value_col)
    else:
        print(f"❌ Error: Unknown chart type")
        return
    
    print(f"✅ Transformed for {chart_type} chart")
    
    # Step 4: Display!
    print("\nStep 4: Generating dashboard...")
    print("="*60)
    tools.display_dashboard(chart_data, title)

In [ ]:
# Test 1: Natural language to bar chart
create_dashboard_from_nl(
    "Show me the total quantity sold for each product",
    chart_type="bar",
    title="Total Sales by Product"
)

In [ ]:
# Test 2: Natural language to pie chart
create_dashboard_from_nl(
    "What's the breakdown of sales by product category?",
    chart_type="pie",
    title="Sales Distribution by Category"
)

### Mini-Exercise: Business Insights Challenge

Now it's your turn! The VP of Sales needs two quick insights for a meeting. Use the pipeline to answer these business questions:

In [ ]:
# Challenge 1: The VP wants to know which customers are driving the most revenue
# Write a natural language question to find total spending per customer

create_dashboard_from_nl(
    "???",  # TODO: Ask about customer spending/revenue
    chart_type="bar",
    title="Customer Revenue Analysis"
)

In [ ]:
# Challenge 2: The VP wants to see sales trends to plan inventory
# Write a natural language question to show monthly sales over time
# Hint: What chart is most appropriate to display time-series data!

create_dashboard_from_nl(
    "???",  # TODO: Ask about sales over time (monthly trend)
    chart_type="???", 
    title="Monthly Sales Trend"
)

---
## 3.5: Design Discussion - Why Not Use an Agent?

### The Elephant in the Room

Look at our `create_dashboard_from_nl()` function. Notice something interesting?

```python
def create_dashboard_from_nl(user_question, chart_type="bar", title="Dashboard"):
    sql_query = tools.nl_to_sql(user_question)      # Step 1: Always happens
    query_result = tools.execute_sql_query(...)      # Step 2: Always happens
    chart_data = tools.transform_for_bar_chart(...)  # Step 3: Always happens
    tools.display_dashboard(chart_data, title)       # Step 4: Always happens
```

This is a **deterministic pipeline** - the steps are fixed and always execute in the same order. An **Agent**, on the other hand, would use an LLM to decide which tool to call at each step.

### The Big Question

**Why didn't we use an Agent to orchestrate the dashboard pipeline?**

In the exercise below, evaluate whether each statement is a **valid reason** for our design choice, or a **misconception** about Agents vs Deterministic pipelines.

In [ ]:
# Exercise: Agent vs Deterministic Pipeline
# For each statement, mark whether it's a VALID reason or a MISCONCEPTION

# Replace "???" with "VALID" or "MISCONCEPTION"
design_choices = {
    "1. Deterministic pipelines are faster because they don't need LLM calls for orchestration": "???",
    "2. Deterministic pipelines are cheaper because they use fewer LLM API calls": "???",
    "3. Our pipeline steps always happen in the same order, so an Agent would add unnecessary complexity / require an involved agentic loop": "???",
    "4. Agents can't work with databases - only deterministic code can": "???",
    "5. Deterministic pipelines are easier to debug and test": "???",
    "6. An Agent would be useful if users asked open-ended questions like 'What insights can you find about sales?'": "???",
}

# Print your answers
print("="*70)
print("Agent vs Deterministic Pipeline - Design Analysis")
print("="*70)
for statement, answer in design_choices.items():
    print(f"\n{statement}")
    print(f"   Your answer: {answer}")

In [ ]:
# Run this cell to check your answers
tools.verify_agent_vs_deterministic(design_choices)

---
# Summary

## What We Learned Today

### Part 1: Tool Identification
- **Core Concept**: LLM + Tool = Agentic AI
- **Workflow Identification**: Recognizing where agentic AI can add value
- **Function Calling**: Standardized way to describe tools to LLMs
- **Tool Adapters**: Bridging LLM requests with actual tool execution

### Part 2: Orchestration 
- Simple tools are **fragile** - they fail on typos, case differences, ambiguous queries
- LLMs can **hallucinate** parameters that don't exist
- Production systems need **multi-step validation** and **schema-awareness**

### Part 3: Dashboard Builder
- **Schema Understanding**: Know your data before building dashboards
- **Data Transformation**: Bridge the gap between SQL results and chart formats
- **Visualization**: Turn data into beautiful, actionable insights
- **Pipeline Design**: When to use deterministic pipelines vs agents

## Key Takeaways

1. **Agentic AI** enables autonomous task completion through tool use
2. **Proper tool definitions** are crucial for LLM understanding
3. **Schema awareness** prevents impossible queries and hallucinations
4. **Design principle**: Start simple (deterministic), add Agents only when flexibility is needed

## What's Next?

You now have the building blocks to create your own agentic systems:
- Connect to your own databases
- Define custom transformation functions
- Build domain-specific visualizations
- Add more sophisticated orchestration patterns when needed

**Congratulations on completing Day 2!**